In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image

# BASE_PATH = "/lustre/fswork/projects/rech/rbw/ucw75ke/projects/GradientDistillation"
BASE_PATH = "/Users/alex/Developpement/Internship/GradientDistillation"
BACKBONE = "dinov2_vitb"
run_name = "dinov2_vitb_latent_slurpp_medoids_s3407"

data = torch.load(
    f"{BASE_PATH}/logged_files/distillation/aqua20/{BACKBONE}/{run_name}/data.pth",
    weights_only=False, map_location="cpu",
)
print("clés data.pth:", list(data.keys()))

In [ ]:
# ---- récupération défensive : le nom de la clé de snapshots dépend du logger
snap_key = next((k for k in data if "snapshot" in k and "step" not in k), None)
step_key = next((k for k in data if "snapshot" in k and "step" in k), None)
snapshots = data.get(snap_key, [])
steps = data.get(step_key, list(range(len(snapshots))))
print(f"{len(snapshots)} snapshots ({snap_key}), steps: {steps}")
if snapshots:
    print("clés snapshot:", snapshots[0].keys())
    print("shape I:", snapshots[0]["I"].shape)

# pred_latent est dans save_dict (get_to_save), pas dans les snapshots
sd = data.get("save_dict", data)
pred_latent = sd["pred_latent"].float().cpu()      # [N, 12, r, r]
latent_prior = sd["latent_prior"].float().cpu()
sample_paths = sd.get("sample_paths", None)
print("pred_latent:", tuple(pred_latent.shape))

COMPS = {"clear (J)": slice(0, 4), "bc (B)": slice(4, 8), "ill (T)": slice(8, 12)}

In [ ]:
def show_evolution(snapshots, steps, key="I", class_indices=None, figsize_per_img=1.5):
    """Évolution temporelle des sorties décodées. key: 'I', 'J', 'T', 'B'."""
    N = snapshots[0][key].shape[0]
    cols = list(class_indices) if class_indices is not None else list(range(N))
    fig, axes = plt.subplots(len(snapshots), len(cols),
                             figsize=(len(cols) * figsize_per_img,
                                      len(snapshots) * figsize_per_img),
                             squeeze=False)
    for i, (snap, step) in enumerate(zip(snapshots, steps)):
        imgs = snap[key]
        for j, c in enumerate(cols):
            axes[i][j].imshow(imgs[c].clamp(0, 1).permute(1, 2, 0).cpu())
            axes[i][j].axis("off")
        axes[i][0].set_ylabel(f"it {step}", rotation=0, labelpad=30)
        axes[i][0].axis("on")
        axes[i][0].set_xticks([]); axes[i][0].set_yticks([])
    fig.suptitle(key)
    plt.tight_layout(); plt.show()


if snapshots:
    for k in ["I", "J", "T", "B"]:
        show_evolution(snapshots, steps, key=k)

In [ ]:
def show_latent_channels(latent, class_idx=0, figsize_per_img=1.4):
    """Les 12 canaux latents d'une image, groupés par composante physique."""
    z = latent[class_idx]                       # [12, r, r]
    fig, axes = plt.subplots(3, 4, figsize=(4 * figsize_per_img, 3 * figsize_per_img),
                             squeeze=False)
    for row, (name, sl) in enumerate(COMPS.items()):
        block = z[sl]
        vmax = block.abs().max()
        for c in range(4):
            axes[row][c].imshow(block[c], cmap="RdBu_r", vmin=-vmax, vmax=vmax)
            axes[row][c].axis("off")
        axes[row][0].set_ylabel(name, rotation=0, labelpad=35, fontsize=8)
        axes[row][0].axis("on")
        axes[row][0].set_xticks([]); axes[row][0].set_yticks([])
    fig.suptitle(f"pred_latent — classe {class_idx}")
    plt.tight_layout(); plt.show()


show_latent_channels(pred_latent, class_idx=0)

In [ ]:
def show_drift_maps(pred, prior, class_indices=None, figsize_per_img=1.4):
    """Carte spatiale |pred - prior| par composante : où le latent a bougé."""
    N = pred.shape[0]
    cols = list(class_indices) if class_indices is not None else list(range(N))
    d = (pred - prior).abs()
    fig, axes = plt.subplots(3, len(cols),
                             figsize=(len(cols) * figsize_per_img, 3 * figsize_per_img),
                             squeeze=False)
    for row, (name, sl) in enumerate(COMPS.items()):
        m = d[:, sl].mean(1)                     # [N, r, r]
        vmax = m.max()
        for j, c in enumerate(cols):
            axes[row][j].imshow(m[c], cmap="magma", vmin=0, vmax=vmax)
            axes[row][j].axis("off")
        axes[row][0].set_ylabel(name, rotation=0, labelpad=35, fontsize=8)
        axes[row][0].axis("on")
        axes[row][0].set_xticks([]); axes[row][0].set_yticks([])
    fig.suptitle("dérive latente |pred − prior| (échelle commune par composante)")
    plt.tight_layout(); plt.show()


show_drift_maps(pred_latent, latent_prior)

In [ ]:
def drift_per_class(pred, prior, class_names=None):
    """RMSE de dérive par classe et par composante — l'analogue latent du 'T freeze'."""
    fig, ax = plt.subplots(figsize=(10, 3.2))
    N = pred.shape[0]
    x = np.arange(N)
    w = 0.27
    for k, (name, sl) in enumerate(COMPS.items()):
        r = torch.sqrt(((pred[:, sl] - prior[:, sl]) ** 2).mean(dim=(1, 2, 3)))
        ax.bar(x + (k - 1) * w, r.numpy(), width=w, label=name)
        print(f"{name:>10}  drift RMSE moyen = {r.mean():.4f}  "
              f"(relatif = {(r.mean() / prior[:, sl].std()):.3%})")
    ax.set_xticks(x)
    ax.set_xticklabels(class_names if class_names else x, rotation=60,
                       ha="right", fontsize=7)
    ax.set_ylabel("RMSE latente"); ax.legend(fontsize=8)
    ax.set_title("Dérive par rapport à l'initialisation SLURPP")
    plt.tight_layout(); plt.show()


drift_per_class(pred_latent, latent_prior)

In [ ]:
def latent_distributions(pred, prior):
    """Le latent reste-t-il dans le support du VAE ? (dérive vs saturation)"""
    fig, axes = plt.subplots(1, 3, figsize=(12, 3), squeeze=False)
    for k, (name, sl) in enumerate(COMPS.items()):
        ax = axes[0][k]
        ax.hist(prior[:, sl].flatten().numpy(), bins=80, alpha=0.5,
                density=True, label="prior")
        ax.hist(pred[:, sl].flatten().numpy(), bins=80, alpha=0.5,
                density=True, label="optimisé")
        ax.set_title(name, fontsize=9); ax.legend(fontsize=7)
    fig.suptitle("distribution des valeurs latentes")
    plt.tight_layout(); plt.show()


latent_distributions(pred_latent, latent_prior)

In [ ]:
def compare_to_medoids(snapshots, sample_paths, class_indices=None, figsize_per_img=1.6):
    """Médoïde source (haut) vs I distillé final (bas) : ce que l'optimisation a changé."""
    if not snapshots or sample_paths is None:
        print("snapshots ou sample_paths absents"); return
    I = snapshots[-1]["I"]
    cols = list(class_indices) if class_indices is not None else list(range(I.shape[0]))
    fig, axes = plt.subplots(2, len(cols),
                             figsize=(len(cols) * figsize_per_img, 2 * figsize_per_img),
                             squeeze=False)
    for j, c in enumerate(cols):
        axes[0][j].imshow(Image.open(sample_paths[c]).convert("RGB").resize((256, 256)))
        axes[1][j].imshow(I[c].clamp(0, 1).permute(1, 2, 0))
        axes[0][j].axis("off"); axes[1][j].axis("off")
    for row, lab in enumerate(["médoïde", "distillé"]):
        axes[row][0].set_ylabel(lab, rotation=0, labelpad=30, fontsize=8)
        axes[row][0].axis("on")
        axes[row][0].set_xticks([]); axes[row][0].set_yticks([])
    plt.tight_layout(); plt.show()


compare_to_medoids(snapshots, sample_paths)

In [ ]:

def residual_vs_medoid(snapshots, sample_paths, class_indices=None, figsize_per_img=1.6):
    """|I_final − médoïde| en pixel : localise le signal ajouté par la distillation."""
    if not snapshots or sample_paths is None:
        return
    I = snapshots[-1]["I"]
    R = I.shape[-1]
    cols = list(class_indices) if class_indices is not None else list(range(I.shape[0]))
    fig, axes = plt.subplots(1, len(cols),
                             figsize=(len(cols) * figsize_per_img, figsize_per_img),
                             squeeze=False)
    for j, c in enumerate(cols):
        ref = np.asarray(Image.open(sample_paths[c]).convert("RGB")
                         .resize((R, R)), dtype=np.float32) / 255.0
        res = (I[c].clamp(0, 1).permute(1, 2, 0).numpy() - ref)
        axes[0][j].imshow(np.abs(res).mean(-1), cmap="magma", vmin=0, vmax=0.5)
        axes[0][j].axis("off")
    fig.suptitle("|I distillé − médoïde|")
    plt.tight_layout(); plt.show()


residual_vs_medoid(snapshots, sample_paths)